## Config

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import deque
import time

# --- Configuration ---
VIDEO_SOURCE = r"TULT-4.mp4"
SLOT_MODEL_PATH = r"weights\parking-obb2\weights\best.pt"
CAR_MODEL_PATH = r"weights\yolo11m-obb.pt"

WINDOW_NAME = "Parking Detection"
WINDOW_WIDTH = 1024
WINDOW_HEIGHT = 576

# --- Load Models ---
print("Loading YOLO models...")
slot_model = YOLO(SLOT_MODEL_PATH)
car_model = YOLO(CAR_MODEL_PATH)

# --- Print Model Class Names ---
print("\nYOLO model class names (car_model):")
for i, name in car_model.names.items():
    print(f"{i}: {name}")

print("\nYOLO model class names (slot_model):")
for i, name in slot_model.names.items():
    print(f"{i}: {name}")

# --- Optional: Verify video file ---
cap = cv2.VideoCapture(VIDEO_SOURCE)
if not cap.isOpened():
    print(f"\nError: Could not open video source: {VIDEO_SOURCE}")
else:
    print(f"\nVideo source loaded: {VIDEO_SOURCE}")
    cap.release()


Loading YOLO models...

YOLO model class names (car_model):
0: plane
1: ship
2: storage tank
3: baseball diamond
4: tennis court
5: basketball court
6: ground track field
7: harbor
8: bridge
9: large vehicle
10: small vehicle
11: helicopter
12: roundabout
13: soccer ball field
14: swimming pool

YOLO model class names (slot_model):
0: space-empty
1: space-occupied

Video source loaded: TULT-4.mp4


## Calibration

In [2]:
# --- Phase 1: Parking Spot Calibration Settings ---
SLOT_CONF_THRESH = 0.55  # Confidence threshold for a parking spot detection
CALIBRATION_FRAMES = 60  # Number of frames to run to define spots
STABILITY_THRESHOLD = (CALIBRATION_FRAMES * 0.25)  # A spot must be seen in at least this many frames to be kept
CENTROID_DISTANCE_THRESH = 20  # Max distance between centroids to be considered the same spot

# --- Phase 2: Occupancy Detection Settings ---
CAR_CONF_THRESH = 0.5
IMG_SIZE = 640

# --- Helper Function ---
def get_polygon_centroid(poly):
    return np.mean(poly, axis=0).astype(int)

# Get the class ID for vehicle from the standard model's names
CAR_CLASS_ID = -1
for class_id, name in car_model.names.items():
    if name == 'large vehicle' or name == 'car' or name == 'truck' or name == 'bus':
        CAR_CLASS_ID = class_id
        break
if CAR_CLASS_ID == -1:
    raise ValueError("Could not find 'car/vehicle' class in the detection model.")

print(f"Parking Slot Model Loaded. Detection Model Loaded (Class ID: {CAR_CLASS_ID}).")

Parking Slot Model Loaded. Detection Model Loaded (Class ID: 9).


In [3]:
# --- 2. Calibration Phase: Define Parking Slots (with Visualization) ---
print("\n--- Starting Calibration Phase ---")
cap = cv2.VideoCapture(VIDEO_SOURCE)
if not cap.isOpened():
    raise IOError(f"Cannot open video source: {VIDEO_SOURCE}")

cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, WINDOW_WIDTH, WINDOW_HEIGHT)

potential_slots = []
slot_id_counter = 0
frame_count = 0

# --- UI parameters ---
bar_height = 25
bar_color_bg = (40, 40, 40)
bar_color_fg = (0, 255, 255)
text_color = (255, 255, 255)
font = cv2.FONT_HERSHEY_SIMPLEX

while frame_count < CALIBRATION_FRAMES:
    ret, frame = cap.read()
    if not ret:
        break

    # --- Run slot detection model ---
    results = slot_model(frame, imgsz=IMG_SIZE, verbose=False, conf=SLOT_CONF_THRESH)
    boxes = results[0].obb

    if boxes is not None:
        for box in boxes:
            poly = box.xyxyxyxy[0].cpu().numpy().astype(int)
            centroid = get_polygon_centroid(poly)
            
            # Match to existing potential slots
            best_match_idx = -1     # Placeholder for best match index
            min_dist = float('inf')     # Initialize minimum distance to infinity
            for i, slot in enumerate(potential_slots):      # Iterate over existing slots
                dist = np.linalg.norm(centroid - slot['centroid']) # Calculate distance between centroids
                if dist < CENTROID_DISTANCE_THRESH and dist < min_dist: 
                    min_dist = dist 
                    best_match_idx = i 

            if best_match_idx != -1:
                # Update matched slot
                matched_slot = potential_slots[best_match_idx]
                matched_slot['polygon'] = (matched_slot['polygon'] * matched_slot['frames_seen'] + poly) / (matched_slot['frames_seen'] + 1)
                matched_slot['centroid'] = get_polygon_centroid(matched_slot['polygon'])
                matched_slot['frames_seen'] += 1
            else:
                # Add new potential slot
                potential_slots.append({
                    'id': slot_id_counter,
                    'polygon': poly,
                    'centroid': centroid,
                    'frames_seen': 1
                })
                slot_id_counter += 1

    # --- Visualization Section ---
    progress_ratio = (frame_count + 1) / CALIBRATION_FRAMES
    progress_percent = int(progress_ratio * 100)

    # Draw polygons
    for spot in potential_slots:
        poly = spot['polygon'].astype(int)
        seen = spot['frames_seen']
        color = (0, min(255, seen * 25), 255 - min(255, seen * 25))  # from yellow to green as it stabilizes
        cv2.polylines(frame, [poly], True, color, 2)
        cv2.circle(frame, tuple(spot['centroid'].astype(int)), 3, color, -1)
        cv2.putText(frame, f"ID:{spot['id']}", tuple(spot['centroid'].astype(int) + 10),
                    font, 0.5, color, 1, cv2.LINE_AA)

    # Progress bar background
    cv2.rectangle(frame, (10, 10), (frame.shape[1] - 10, 10 + bar_height), bar_color_bg, -1)
    cv2.rectangle(frame, (10, 10), (10 + int(progress_ratio * (frame.shape[1] - 20)), 10 + bar_height), bar_color_fg, -1)

    # Text overlay
    cv2.rectangle(frame, (5, 50), (290, 125), (0, 0, 0), -1)

    cv2.putText(frame, f"Calibration: {progress_percent}%", (10, 80),
                font, 0.9, text_color, 2, cv2.LINE_AA)
    cv2.putText(frame, f"Detected Zones: {len(potential_slots)}", (10, 115),
                font, 0.8, (0, 255, 255), 2, cv2.LINE_AA)

    cv2.imshow(WINDOW_NAME, frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

    frame_count += 1
    time.sleep(0.01)

# --- Filter out unstable slots ---
stable_parking_slots = [slot for slot in potential_slots if slot['frames_seen'] >= STABILITY_THRESHOLD]
print(f"--- Calibration Complete: {len(stable_parking_slots)} stable parking slots defined. ---")

# --- Export results ---
import json, os

OUTPUT_DIR = "exported_detections"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def poly_to_list(poly):
    return [[int(x), int(y)] for x, y in poly]

# Save final detected parking slots
export_data = []
for s in stable_parking_slots:
    export_data.append({
        "id": int(s["id"]),
        "polygon": poly_to_list(s["polygon"]),
        "centroid": [int(s["centroid"][0]), int(s["centroid"][1])],
        "frames_seen": int(s["frames_seen"])
    })

json_path = os.path.join(OUTPUT_DIR, "stable_parking_slots.json")
with open(json_path, "w") as f:
    json.dump(export_data, f, indent=2)

print(f"✅ Exported {len(export_data)} parking slots to {json_path}")

# --- Visualization of Final Result ---
ret, final_frame = cap.read()
if ret:
    for slot in stable_parking_slots:
        cv2.polylines(final_frame, [slot['polygon'].astype(int)], True, (0, 255, 0), 2)
        cv2.circle(final_frame, tuple(slot['centroid'].astype(int)), 4, (0, 255, 0), -1)
        cv2.putText(final_frame, f"P{slot['id']}", tuple(slot['centroid'].astype(int) + 10),
                    font, 0.6, (0, 255, 0), 2, cv2.LINE_AA)

    overlay_text = f"Calibration Complete: {len(stable_parking_slots)} Stable Zones"
    (tw, th), _ = cv2.getTextSize(overlay_text, font, 1.0, 2)
    cv2.rectangle(final_frame, (10, 10), (30 + tw, 30 + th), (0, 0, 0), -1)
    cv2.putText(final_frame, overlay_text, (20, 40), font, 1.0, (0, 255, 0), 2, cv2.LINE_AA)

    cv2.imshow(WINDOW_NAME, final_frame)
    cv2.waitKey(10)


--- Starting Calibration Phase ---
--- Calibration Complete: 74 stable parking slots defined. ---
✅ Exported 74 parking slots to exported_detections\stable_parking_slots.json


# Result Correction

In [6]:
import cv2
import json
import numpy as np

VIDEO_SOURCE = "TULT-4.mp4"
INPUT_JSON = "exported_detections/stable_parking_slots.json"
OUTPUT_JSON = "exported_detections/stable_parking_slots_corrected.json"
WINDOW_NAME_COR = "Edit Parking Slots"

# --- Load first video frame for background ---
cap = cv2.VideoCapture(VIDEO_SOURCE)
ret, background = cap.read()
if not ret:
    raise IOError(f"Cannot read video: {VIDEO_SOURCE}")

cv2.resizeWindow(WINDOW_NAME_COR, WINDOW_WIDTH, WINDOW_HEIGHT)
cap.release()

# --- Load polygons ---
with open(INPUT_JSON, "r") as f:
    slots = json.load(f)
for s in slots:
    s["polygon"] = np.array(s["polygon"], dtype=np.int32)

# --- Editor state ---
selected_slot = None
selected_vertex = None
mode = "move"  # move / add / delete / new
new_poly_points = []
font = cv2.FONT_HERSHEY_SIMPLEX

VERTEX_RADIUS = 6
SELECT_DIST = 12

def draw_control_bar(frame, mode):
    overlay = frame.copy()
    h, w, _ = frame.shape

    # --- Compact semi-transparent top bar ---
    bar_h = 35
    cv2.rectangle(overlay, (0, 0), (w, bar_h), (0, 0, 0), -1)
    frame = cv2.addWeighted(overlay, 0.5, frame, 0.5, 0)

    # --- Text content (short and clear) ---
    text = (
        f"[M] Move  [A] AddVtx  [D] DelVtx  [N] NewSlot  "
        f"[DEL] DelSlot  [S] Save  [Q] Quit   Mode: {mode.upper()}"
    )
    cv2.putText(frame, text, (10, 24), font, 0.6, (255, 255, 255), 1, cv2.LINE_AA)
    return frame

def draw():
    frame = background.copy()
    for i, slot in enumerate(slots):
        poly = slot["polygon"]
        color = (0, 255, 0) if i != selected_slot else (0, 255, 255)
        if len(poly) >= 3:
            cv2.polylines(frame, [poly], True, color, 2)
        for (x, y) in poly:
            cv2.circle(frame, (x, y), VERTEX_RADIUS, color, -1)
        cv2.putText(frame, f"P{slot['id']}", tuple(poly[0] + np.array([5, -5])), font, 0.6, color, 2)

    if mode == "new" and new_poly_points:
        pts = np.array(new_poly_points, np.int32)
        cv2.polylines(frame, [pts], False, (255, 255, 0), 2)
        for (x, y) in pts:
            cv2.circle(frame, (x, y), VERTEX_RADIUS, (255, 255, 0), -1)

    frame = draw_control_bar(frame, mode)
    return frame

def find_vertex(x, y):
    for si, slot in enumerate(slots):
        for vi, (vx, vy) in enumerate(slot["polygon"]):
            if (vx - x)**2 + (vy - y)**2 < SELECT_DIST**2:
                return si, vi
    return None, None

def find_nearest_edge(point, poly):
    px, py = point
    min_dist = float("inf")
    insert_idx = 0
    for i in range(len(poly)):
        x1, y1 = poly[i]
        x2, y2 = poly[(i + 1) % len(poly)]
        a = np.array([x1, y1])
        b = np.array([x2, y2])
        p = np.array([px, py])
        ab = b - a
        t = np.clip(np.dot(p - a, ab) / (np.dot(ab, ab) + 1e-6), 0, 1)
        proj = a + t * ab
        dist = np.linalg.norm(p - proj)
        if dist < min_dist:
            min_dist = dist
            insert_idx = i + 1
    return insert_idx

def mouse_callback(event, x, y, flags, param):
    global selected_slot, selected_vertex, new_poly_points, mode

    if event == cv2.EVENT_LBUTTONDOWN:
        si, vi = find_vertex(x, y)

        if mode == "move":
            if si is not None:
                selected_slot = si
                selected_vertex = vi
            else:
                dists = [np.linalg.norm(np.mean(s["polygon"], axis=0) - np.array([x, y])) for s in slots]
                if dists:
                    selected_slot = int(np.argmin(dists))
        elif mode == "add" and selected_slot is not None:
            poly = slots[selected_slot]["polygon"]
            insert_idx = find_nearest_edge((x, y), poly)
            slots[selected_slot]["polygon"] = np.insert(poly, insert_idx, [x, y], axis=0)
        elif mode == "delete" and si is not None:
            if len(slots[si]["polygon"]) > 3:
                slots[si]["polygon"] = np.delete(slots[si]["polygon"], vi, axis=0)
        elif mode == "new":
            new_poly_points.append([x, y])
            if len(new_poly_points) >= 4:
                new_id = max([s["id"] for s in slots]) + 1 if slots else 0
                slots.append({
                    "id": new_id,
                    "polygon": np.array(new_poly_points, dtype=np.int32),
                    "centroid": [int(np.mean([p[0] for p in new_poly_points])),
                                 int(np.mean([p[1] for p in new_poly_points]))]
                })
                print(f"🆕 Added new parking slot ID {new_id}")
                new_poly_points.clear()
                mode = "move"

    elif event == cv2.EVENT_MOUSEMOVE and selected_vertex is not None:
        slots[selected_slot]["polygon"][selected_vertex] = [x, y]

    elif event == cv2.EVENT_LBUTTONUP:
        selected_vertex = None

cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.setMouseCallback(WINDOW_NAME, mouse_callback)

while True:
    img = draw()
    cv2.imshow(WINDOW_NAME, img)
    key = cv2.waitKey(30) & 0xFF

    if key == ord("q"):
        break
    elif key == ord("m"):
        mode = "move"
    elif key == ord("a"):
        mode = "add"
    elif key == ord("d"):
        mode = "delete"
    elif key == ord("n"):
        mode = "new"
        new_poly_points.clear()
    elif key in [127, 8]:  # Delete key
        if selected_slot is not None:
            deleted_id = slots[selected_slot]["id"]
            slots.pop(selected_slot)
            print(f"❌ Deleted parking slot ID {deleted_id}")
            selected_slot = None
    elif key == ord("s"):
        export_data = []
        for s in slots:
            poly = s["polygon"].astype(int)
            cent = np.mean(poly, axis=0).astype(int).tolist()
            export_data.append({
                "id": int(s["id"]),
                "polygon": poly.tolist(),
                "centroid": cent
            })
        with open(OUTPUT_JSON, "w") as f:
            json.dump(export_data, f, indent=2)
        print(f"✅ Saved corrected slots to {OUTPUT_JSON}")

cv2.destroyAllWindows()


🆕 Added new parking slot ID 74
🆕 Added new parking slot ID 75
🆕 Added new parking slot ID 76
🆕 Added new parking slot ID 77
✅ Saved corrected slots to exported_detections/stable_parking_slots_corrected.json


## YOLO BASE MODEL

In [ ]:
# --- 3. Main Detection Loop: Check Occupancy ---

CAR_MODEL_PATH = r"weights\yolo11m.pt"
car_model = YOLO(CAR_MODEL_PATH)

print("\n--- Starting Occupancy Detection Phase ---")
cap.set(cv2.CAP_PROP_POS_FRAMES, 0) # Reset video to the beginning

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Initialize all slots as empty for the current frame
    slot_statuses = {slot['id']: 'Empty' for slot in stable_parking_slots}
    occupied_count = 0

    # Run car detection
    car_results = car_model(frame, classes=[CAR_CLASS_ID], imgsz=IMG_SIZE, verbose=False, conf=CAR_CONF_THRESH)
    
    # Check each detected car against the stable parking slots
    for car_box in car_results[0].boxes:
        x1, y1, x2, y2 = car_box.xyxy[0].cpu().numpy().astype(int)
        
        # --- Draw the bounding box for the detected car ---
        cv2.rectangle(frame, (x1, y1), (x2, y2), (255, 0, 0), 2) # Draws a blue box

        # Use the bottom center of the car's bounding box as its location point
        car_center_point = (int((x1 + x2) / 2), int((y1 + y2) / 2))

        for slot in stable_parking_slots:
            contour = slot['polygon'].astype(np.float32)
            # Check if the car's point is inside the slot's polygon
            if cv2.pointPolygonTest(contour, car_center_point, False) >= 0:
                slot_statuses[slot['id']] = 'Occupied'
                break # A car can only be in one spot

    # --- Drawing and Display ---
    empty_count = len(stable_parking_slots)
    for slot in stable_parking_slots:
        poly = slot['polygon'].astype(int)
        status = slot_statuses[slot['id']]
        
        if status == 'Occupied':
            color = (0, 0, 255) # Red for occupied
            occupied_count += 1
        else:
            color = (0, 255, 0) # Green for empty

        cv2.polylines(frame, [poly], isClosed=True, color=color, thickness=2)
        # Put the spot ID text near its centroid
        cv2.putText(frame, f"P{slot['id']}", (slot['centroid'][0]-10, slot['centroid'][1]+5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
    
    empty_count -= occupied_count

    # --- Overlay Stats ---
    y_offset = 30
    stats_text_occupied = f"Occupied: {occupied_count}"
    stats_text_empty = f"Empty: {empty_count}"
    total_text = f"Total Spaces: {len(stable_parking_slots)}"

    for i, text in enumerate([stats_text_occupied, stats_text_empty, total_text]):
        color = (0, 0, 255) if i == 0 else (0, 255, 0) if i == 1 else (0, 255, 255)
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
        cv2.rectangle(frame, (5, y_offset - th - 5), (15 + tw, y_offset + 5), (0, 0, 0), -1)
        cv2.putText(frame, text, (10, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        y_offset += 40

    # Show the frame
    cv2.imshow("Parking Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

# --- Cleanup ---
cap.release()
cv2.destroyAllWindows()
print("--- Script Finished ---")

## YOLO OBB MODEL

In [ ]:
# --- 3. Main Detection Loop: Check Occupancy (OBB Model) ---
# --- Sort parking slots by their position on screen (top-left to bottom-right) ---

CAR_MODEL_PATH = r"weights\yolo11m-obb.pt"
car_model = YOLO(CAR_MODEL_PATH)

stable_parking_slots = sorted(
    stable_parking_slots,
    key=lambda z: (z['centroid'][1], z['centroid'][0])  # first by Y (top to bottom), then X (left to right)
)

# --- Reassign slot IDs in that order ---
for idx, slot in enumerate(stable_parking_slots, start=1):
    slot['id'] = idx

print("\n--- Starting Occupancy Detection Phase (OBB Model) ---")
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)  # Reset video

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    slot_statuses = {slot['id']: 'Empty' for slot in stable_parking_slots}
    occupied_count = 0

    # --- Run car detection (OBB version) ---
    car_results = car_model(frame, classes=[CAR_CLASS_ID], imgsz=IMG_SIZE, verbose=False, conf=CAR_CONF_THRESH)
    obb_boxes = car_results[0].obb

    if obb_boxes is not None:
        for obb in obb_boxes:
            conf = float(obb.conf[0])
            if conf < CAR_CONF_THRESH:
                continue

            # --- Get the 4 corner points of the oriented box ---
            poly = obb.xyxyxyxy[0].cpu().numpy().astype(int).reshape(-1, 2)

            # Draw rotated polygon for visualization
            cv2.polylines(frame, [poly], isClosed=True, color=(255, 0, 0), thickness=2)

            # Compute bottom-center point of the rotated box (approximate "car position")
            mid_y = int(np.mean(poly[:, 1]))
            mid_x = int(np.mean(poly[:, 0]))
            car_center_point = (mid_x, mid_y)

            # Check if the car center is inside any parking slot polygon
            for slot in stable_parking_slots:
                contour = slot['polygon'].astype(np.float32)
                if cv2.pointPolygonTest(contour, car_center_point, False) >= 0:
                    slot_statuses[slot['id']] = 'Occupied'
                    break  # stop checking once matched

    # --- Draw Parking slot Polygons ---
    empty_count = len(stable_parking_slots)
    for slot in stable_parking_slots:
        poly = slot['polygon'].astype(int)
        status = slot_statuses[slot['id']]

        if status == 'Occupied':
            color = (0, 0, 255)  # red
            occupied_count += 1
        else:
            color = (0, 255, 0)  # green

        cv2.polylines(frame, [poly], isClosed=True, color=color, thickness=2)
        cv2.putText(frame, f"P{slot['id']}",
                    (slot['centroid'][0]-10, slot['centroid'][1]+5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)

    empty_count -= occupied_count

    # --- Overlay Statistics ---
    y_offset = 30
    stats = [
        ("Occupied", occupied_count, (0, 0, 255)),
        ("Empty", empty_count, (0, 255, 0)),
        ("Total Spaces", len(stable_parking_slots), (0, 255, 255))
    ]

    for label, value, color in stats:
        text = f"{label}: {value}"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
        cv2.rectangle(frame, (5, y_offset - th - 5), (15 + tw, y_offset + 5), (0, 0, 0), -1)
        cv2.putText(frame, text, (10, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        y_offset += 40

    # --- Display frame ---
    cv2.imshow("Parking Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
print("--- Script Finished ---")


## CUSTOM TRAIN YOLO OBB

In [ ]:
# --- 3. Main Detection Loop: Check Occupancy (Single OBB Model) ---
# --- Sort parking zones by position (top-left to bottom-right) ---
stable_parking_slots = sorted(
    stable_parking_slots,
    key=lambda z: (z['centroid'][1], z['centroid'][0])
)

# --- Reassign slot IDs in sorted order ---
for idx, slot in enumerate(stable_parking_slots, start=1):
    slot['id'] = idx

print("\n--- Starting Parking Occupancy Detection (SLOT MODEL) ---")
cap.set(cv2.CAP_PROP_POS_FRAMES, 0)

# --- Parameters for optimization ---
FRAME_SKIP = 4          # process every 4th frame (change to 5 for more speed)
CONF_THRESH = 0.8       # filter low confidence
SMOOTH_WINDOW = 10       # smoothing window (last N frames per slot)

# --- Temporal buffer for smoothing (each slot keeps last N states) ---
slot_history = {s['id']: deque(maxlen=SMOOTH_WINDOW) for s in stable_parking_slots}

fps_time = time.time()
frame_id = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame_id += 1

    # Skip frames for faster real-time behavior
    if frame_id % FRAME_SKIP != 0:
        continue

    # Initialize all slots as "Unknown"
    slot_statuses = {slot['id']: 'Empty' for slot in stable_parking_slots}
    occupied_count = 0

    # --- Run inference with the SLOT model ---
    results = slot_model(frame, verbose=False)[0]
    obb_boxes = results.obb

    if obb_boxes is not None and len(obb_boxes) > 0:
        for obb in obb_boxes:
            cls_id = int(obb.cls[0])
            conf = float(obb.conf[0])
            label = slot_model.names[cls_id].lower()

            # --- Get oriented bounding box polygon ---
            poly = obb.xyxyxyxy[0].cpu().numpy().astype(int).reshape(-1, 2)

            # Compute the centroid of the detected slot
            cx, cy = np.mean(poly[:, 0]), np.mean(poly[:, 1])
            detected_point = (int(cx), int(cy))

            # Draw the rotated bounding box
            # color = (0, 255, 0) if label == "space-empty" else (0, 0, 255)
            # cv2.polylines(frame, [poly], isClosed=True, color=color, thickness=2)
            # cv2.putText(frame, f"{label} ({conf:.2f})",
            #             (int(cx) - 40, int(cy) - 10),
            #             cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2, cv2.LINE_AA)

            # --- Match detection to closest parking zone ---
            for zone in stable_parking_slots:
                contour = zone['polygon'].astype(np.float32)
                if cv2.pointPolygonTest(contour, detected_point, False) >= 0:
                    slot_statuses[zone['id']] = 'Occupied' if label == "space-occupied" else 'Empty'
                    break

        for slot in stable_parking_slots:
            zid = slot['id']
            current_status = slot_statuses[zid]

            # Append the latest frame’s result to the deque
            slot_history[zid].append(current_status)

            # Compute smoothed status based on recent history
            if slot_history[zid].count('Occupied') > SMOOTH_WINDOW // 2:
                slot_statuses[zid] = 'Occupied'
            elif slot_history[zid].count('Empty') > SMOOTH_WINDOW // 2:
                slot_statuses[zid] = 'Empty'

    # --- Draw Parking Slots ---
    total_slots = len(stable_parking_slots)
    empty_count = 0

    for zone in stable_parking_slots:
        poly = zone['polygon'].astype(int)
        status = slot_statuses[zone['id']]

        if status == 'Occupied':
            color = (0, 0, 255)
            occupied_count += 1
        elif status == 'Empty':
            color = (0, 255, 0)
            empty_count += 1
        else:
            color = (128, 128, 128)

        cv2.polylines(frame, [poly], isClosed=True, color=color, thickness=2)
        cv2.putText(frame, f"P{zone['id']}",
                    (zone['centroid'][0] - 10, zone['centroid'][1] + 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)

    # --- Overlay Statistics ---
    y_offset = 30
    stats = [
        ("Occupied", occupied_count, (0, 0, 255)),
        ("Empty", empty_count, (0, 255, 0)),
        ("Total Spaces", total_slots, (0, 255, 255))
    ]

    for label, value, color in stats:
        text = f"{label}: {value}"
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.9, 2)
        cv2.rectangle(frame, (5, y_offset - th - 5), (15 + tw, y_offset + 5), (0, 0, 0), -1)
        cv2.putText(frame, text, (10, y_offset),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)
        y_offset += 40

    # --- Display Frame ---
    cv2.imshow("Parking Detection", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()
print("--- Parking Occupancy Detection Completed ---")
